# PROSIT 2 : Sans complexe

**Groupe :**
* LUU Philippe - Animateur
* AFANE Youcef - Secrétaire
* RABATEL Antonin - Gestionnaire du temps
* RIVET Alexandre - Scribe

---

## 1. Contexte
Dans le cadre d’un projet de tournées de livraison, il est nécessaire de déterminer un itinéraire optimal passant par un ensemble de villes. Contrairement au problème précédent, l’objectif n’est plus de parcourir toutes les routes mais de visiter chaque ville tout en minimisant le temps de trajet total.

Le problème est identifié comme proche du problème du voyageur de commerce, mais nécessite une analyse préalable de sa complexité afin de déterminer s’il est possible de concevoir un algorithme optimal exploitable.

# 2. Mots inconnus / Notions à maîtriser
* **Problème du voyageur de commerce (TSP)** :
Problème consistant à trouver le plus court cycle passant une seule fois par chaque sommet.

* **Problème de décision** :
Version d’un problème dont la réponse est binaire (oui/non), utilisée pour analyser la complexité.

* **Classe NP** :
Ensemble des problèmes dont une solution peut être vérifiée en temps polynomial.

* **NP-Complet** :
Problèmes les plus difficiles de NP, pour lesquels aucun algorithme polynomial n’est connu.

* **Réduction polynomiale** :
Transformation d’un problème en un autre en temps polynomial pour prouver leur complexité équivalente.

* **Heuristique** :
Méthode approchée permettant d’obtenir une solution rapidement sans garantie d’optimalité.

* **Version métrique du TSP** :
Version du TSP respectant l’inégalité triangulaire (distances cohérentes).

# 3. Problématique
Comment modéliser et résoudre un problème d’optimisation de tournée de livraison passant par toutes les villes, sous contrainte de minimisation du temps de trajet, tout en tenant compte de sa complexité algorithmique potentiellement NP-complète ?

# 4. Plan d'action
1. **Modélisation du problème et formulation du problème de décision (1h30)** :
Définir les variables, contraintes et version décisionnelle.
2. **Analyse d’appartenance à la classe NP (1h30)** :
Vérifier qu’une solution peut être validée en temps polynomial.
3. **Étude de la complexité et rapprochement avec le TSP (2h)** :
Comparer avec le problème du voyageur de commerce et identifier sa classe.
4. **Choix d’une stratégie algorithmique adaptée (2h30)** :
Déterminer si une solution exacte ou heuristique est nécessaire.
5. **Validation expérimentale et analyse des performances (2h)** :
Tester les approches sur différentes tailles de graphes.

## Partie 1 : Modélisation du problème

### 1.1 Représentation en graphe

Le réseau de livraison est d'abord modélisé par un **graphe non orienté pondéré** (potentiellement incomplet) :

$$G = (V, E, w)$$

où :
- $V = \{v_0, v_1, \dots, v_{n-1}\}$ est l'ensemble des **sommets** représentant les villes, avec $v_0$ le dépôt de départ
- $E \subseteq \{\{v_i, v_j\} \mid i \neq j\}$ est l'ensemble des **arêtes** représentant les routes existantes
- $w : E \rightarrow \mathbb{R}^+$ est la **fonction de poids**, associant à chaque arête un temps de trajet strictement positif

Le graphe est **non orienté** : $w(\{v_i, v_j\}) = w(\{v_j, v_i\})$, ce qui correspond à des temps de trajet identiques dans les deux sens.

**Complétion du graphe :** le TSP requiert un graphe complet (toutes les paires de villes doivent être reliées). Pour les paires sans route directe, on calcule le **plus court chemin** dans $G$ via l'algorithme de Dijkstra et on l'utilise comme poids de l'arête manquante. On obtient ainsi un graphe complet :

$$G' = (V, E', w')$$

où $E' = \{\{v_i, v_j\} \mid i \neq j\}$ et $w'(\{v_i, v_j\})$ est le coût du plus court chemin entre $v_i$ et $v_j$ dans $G$.

Cette transformation est de complexité $O(n^2 \log n)$, donc **polynomiale**. C'est sur $G'$ que le problème TSP est défini.

---


In [2]:
import heapq

graphe_incomplet = {
    ('A', 'B'): 10,
    ('A', 'C'): 25,
    ('B', 'C'): 12,
    ('B', 'D'): 18,
    ('C', 'D'): 8,
    ('C', 'E'): 15,
    ('D', 'E'): 20,
}

villes = ['A', 'B', 'C', 'D', 'E']

def dijkstra_complet(villes, aretes):
    """Calcule les plus courts chemins entre toutes les paires de sommets."""
    # Construire la liste d'adjacence
    adj = {v: [] for v in villes}
    for (u, v), d in aretes.items():
        adj[u].append((v, d))
        adj[v].append((u, d))
    
    distances = {}
    for source in villes:
        dist = {v: float('inf') for v in villes}
        dist[source] = 0
        file = [(0, source)]
        while file:
            d, u = heapq.heappop(file)
            if d > dist[u]:
                continue
            for voisin, poids in adj[u]:
                nd = d + poids
                if nd < dist[voisin]:
                    dist[voisin] = nd
                    heapq.heappush(file, (nd, voisin))
        for v in villes:
            if source != v:
                cle = (min(source, v), max(source, v))
                if cle not in distances or dist[v] < distances[cle]:
                    distances[cle] = dist[v]
    return distances

graphe_complet = dijkstra_complet(villes, graphe_incomplet)

print("=== Graphe complet (après calcul des plus courts chemins) ===")
print(f"{'Paire':>10} {'Distance directe':>18} {'Plus court chemin':>20}")
print("-" * 52)
for i in range(len(villes)):
    for j in range(i+1, len(villes)):
        paire = (villes[i], villes[j])
        direct = graphe_incomplet.get(paire, graphe_incomplet.get((villes[j], villes[i]), None))
        pcc = graphe_complet[paire]
        directe_str = str(direct) if direct else "pas de route"
        print(f"{villes[i]}-{villes[j]:>8} {directe_str:>18} {pcc:>20}")

print("\n=> Les distances du graphe complet respectent-elles l'inégalité triangulaire ?")
print("   Pour tous sommets i, j, k : d(i,j) <= d(i,k) + d(k,j)")
print("   C'est le cas car on utilise les plus courts chemins !")
print("   => On obtient la version MÉTRIQUE du TSP (Delta-TSP)")

=== Graphe complet (après calcul des plus courts chemins) ===
     Paire   Distance directe    Plus court chemin
----------------------------------------------------
A-       B                 10                   10
A-       C                 25                   22
A-       D       pas de route                   28
A-       E       pas de route                   37
B-       C                 12                   12
B-       D                 18                   18
B-       E       pas de route                   27
C-       D                  8                    8
C-       E                 15                   15
D-       E                 20                   20

=> Les distances du graphe complet respectent-elles l'inégalité triangulaire ?
   Pour tous sommets i, j, k : d(i,j) <= d(i,k) + d(k,j)
   C'est le cas car on utilise les plus courts chemins !
   => On obtient la version MÉTRIQUE du TSP (Delta-TSP)



### 1.2 Hypothèse métrique (Δ-TSP)

Les poids ainsi construits respectent l'**inégalité triangulaire** :

$$\forall\, v_i, v_j, v_k \in V, \quad w(\{v_i, v_k\}) \leq w(\{v_i, v_j\}) + w(\{v_j, v_k\})$$

Cette propriété est **garantie par construction** : un chemin direct ne peut pas être plus long que de passer par une ville intermédiaire (par définition du plus court chemin). Elle est par ailleurs **réaliste** dans le contexte de tournées de livraison.

On se place donc dans la variante dite **TSP métrique** (ou **Δ-TSP**), distincte du TSP général.

---

### 1.3 Problème d'optimisation

**Entrée :** un graphe $G = (V, E, w)$ tel que défini ci-dessus, et un sommet de départ $v_0 \in V$

**Sortie :** un cycle hamiltonien $C = (v_0,\, v_{\sigma(1)},\, v_{\sigma(2)},\, \dots,\, v_{\sigma(n-1)},\, v_0)$ de **coût minimal**, où $\sigma$ est une permutation de $\{1, \dots, n-1\}$

**Objectif :**

$$\min_{\sigma} \sum_{i=0}^{n-1} w\!\left(\left\{v_{\sigma(i)},\, v_{\sigma((i+1) \bmod n)}\right\}\right)$$

Il existe $(n-1)!$ cycles hamiltoniens candidats. Pour $n = 20$ villes, cela représente $\approx 1{,}2 \times 10^{17}$ cycles à évaluer — ce qui motive directement l'étude de la complexité avant toute tentative d'implémentation.

---

### 1.4 Problème de décision associé

La théorie de la complexité s'appuie sur les **problèmes de décision** (réponse oui/non). On reformule le problème d'optimisation ainsi :

> **Étant donné** un graphe $G = (V, E, w)$, un sommet $v_0 \in V$ et un entier $k \in \mathbb{R}^+$, **existe-t-il** un cycle hamiltonien partant de $v_0$ de coût total $\leq k$ ?

**Lien avec l'optimisation :** résoudre le problème d'optimisation revient à trouver la plus petite valeur de $k$ pour laquelle la réponse au problème de décision est « oui ».

C'est ce problème de décision dont on établit la classe de complexité dans les parties suivantes.

## Partie 2 : Appartenance à NP

### Rappel : qu'est-ce que la classe NP ?

Un problème de décision est dans **NP** s'il existe un algorithme (appelé **certificat** ou **vérificateur**) qui :
1. Prend en entrée une **instance** du problème et une **solution candidate**
2. Vérifie en **temps polynomial** si cette solution candidate répond « oui » à l'instance

Pour montrer que notre problème de décision est dans NP, il suffit donc de proposer un tel algorithme certificat et d'analyser sa complexité.

---

### 2.1 Algorithme certificat pour le Δ-TSP

**Entrées du certificat :**
- Le graphe complet $G' = (V, E', w')$
- Un sommet de départ $v_0 \in V$
- Une borne $k \in \mathbb{R}^+$
- Une **solution candidate** : une liste ordonnée de sommets $(s_0, s_1, \dots, s_{n-1})$

**Ce qu'il faut vérifier :**

| Étape | Vérification | Complexité |
|-------|-------------|------------|
| 1 | La solution contient exactement $n$ sommets | $O(1)$ |
| 2 | Le cycle part bien de $v_0$ | $O(1)$ |
| 3 | Chaque sommet de $V$ apparaît exactement une fois (cycle hamiltonien) | $O(n)$ |
| 4 | Le coût total du cycle est $\leq k$ | $O(n)$ |
| **Total** | | $O(n)$ — **polynomial** |

Le certificat s'exécute en $O(n)$, donc en **temps polynomial**.

**Conclusion : le problème de décision du Δ-TSP est dans NP.**

In [3]:
def certificat_tsp(sommets, distances, depart, k, solution_candidate):
    """
    Certificat polynomial pour le problème de décision du Δ-TSP.

    Vérifie si solution_candidate est un cycle hamiltonien
    de coût total <= k, partant de 'depart'.

    Paramètres :
        sommets            : liste des sommets du graphe G'
        distances          : dict {(u, v): poids} pour le graphe complet G'
        depart             : sommet de départ v0
        k                  : borne maximale du coût
        solution_candidate : liste de sommets représentant le cycle proposé

    Retourne : (bool, str) — validité et message explicatif
    """
    n = len(sommets)

    # Étape 1 : vérifier le nombre de sommets — O(1)
    if len(solution_candidate) != n:
        return False, "Nombre de sommets incorrect"

    # Étape 2 : vérifier que le cycle part du bon sommet — O(1)
    if solution_candidate[0] != depart:
        return False, f"Le cycle ne part pas de {depart}"

    # Étape 3 : vérifier que c'est un cycle hamiltonien — O(n)
    # Chaque sommet doit apparaître exactement une fois
    sommets_vus = set()
    for s in solution_candidate:
        if s not in set(sommets):
            return False, f"Le sommet {s} n'existe pas dans le graphe"
        if s in sommets_vus:
            return False, f"Le sommet {s} apparaît plus d'une fois"
        sommets_vus.add(s)

    if sommets_vus != set(sommets):
        manquants = set(sommets) - sommets_vus
        return False, f"Sommets manquants : {manquants}"

    # Étape 4 : calculer le coût total du cycle et vérifier <= k — O(n)
    longueur = 0
    for i in range(n):
        u = solution_candidate[i]
        v = solution_candidate[(i + 1) % n]
        cle = (min(u, v), max(u, v))
        longueur += distances[cle]

    if longueur > k:
        return False, f"Longueur {longueur} > borne {k}"

    return True, f"Cycle hamiltonien valide de longueur {longueur} <= {k}"


# --- Tests ---
# On réutilise le graphe complet G' construit en Partie 1

tests = [
    (['A', 'B', 'D', 'E', 'C'], 100, "cycle valide, borne large"),
    (['A', 'B', 'D', 'E', 'C'], 50,  "cycle valide, borne trop stricte"),
    (['A', 'B', 'C', 'D'],      100, "sommet manquant"),
    (['B', 'A', 'C', 'D', 'E'], 100, "mauvais sommet de départ"),
    (['A', 'B', 'B', 'D', 'E'], 100, "sommet répété"),
]

print("=== Tests de l'algorithme certificat ===\n")
for sol, borne, description in tests:
    valide, msg = certificat_tsp(villes, graphe_complet, 'A', borne, sol)
    status = "  VALIDE" if valide else "INVALIDE"
    print(f"  [{status}] {sol} (k={borne}) — {description}")
    print(f"           -> {msg}\n")

print("=== Analyse de la complexité ===")
print("  Étape 1 : O(1)")
print("  Étape 2 : O(1)")
print("  Étape 3 : O(n)  — parcours de tous les sommets")
print("  Étape 4 : O(n)  — parcours de toutes les arêtes du cycle")
print("  -------")
print("  TOTAL   : O(n)  — POLYNOMIAL")
print("\n=> Le problème de décision du Δ-TSP est dans NP.")

=== Tests de l'algorithme certificat ===

  [  VALIDE] ['A', 'B', 'D', 'E', 'C'] (k=100) — cycle valide, borne large
           -> Cycle hamiltonien valide de longueur 85 <= 100

  [INVALIDE] ['A', 'B', 'D', 'E', 'C'] (k=50) — cycle valide, borne trop stricte
           -> Longueur 85 > borne 50

  [INVALIDE] ['A', 'B', 'C', 'D'] (k=100) — sommet manquant
           -> Nombre de sommets incorrect

  [INVALIDE] ['B', 'A', 'C', 'D', 'E'] (k=100) — mauvais sommet de départ
           -> Le cycle ne part pas de A

  [INVALIDE] ['A', 'B', 'B', 'D', 'E'] (k=100) — sommet répété
           -> Le sommet B apparaît plus d'une fois

=== Analyse de la complexité ===
  Étape 1 : O(1)
  Étape 2 : O(1)
  Étape 3 : O(n)  — parcours de tous les sommets
  Étape 4 : O(n)  — parcours de toutes les arêtes du cycle
  -------
  TOTAL   : O(n)  — POLYNOMIAL

=> Le problème de décision du Δ-TSP est dans NP.


## Partie 3 : Preuve de NP-Complétude du Δ-TSP

On a montré en Partie 2 que le problème de décision du Δ-TSP est **dans NP**. Pour établir sa **NP-Complétude**, il reste à prouver qu'il est **NP-Difficile**, c'est-à-dire au moins aussi difficile que tout problème de NP.

La méthode standard est la **réduction polynomiale** : on prend un problème déjà connu comme NP-Complet et on montre qu'il se transforme en une instance du Δ-TSP en temps polynomial, en conservant la réponse.

---

### 3.1 Rappel : principe de la réduction polynomiale

Pour prouver que le problème B est NP-Complet à partir d'un problème A déjà NP-Complet :

1. Construire une **transformation** $f$ qui transforme toute instance de A en instance de B en **temps polynomial**
2. Prouver la **conservation de la réponse** dans les deux sens :

$$A \text{ répond « oui »} \iff B \text{ répond « oui »}$$

On note $A \leq_p B$ (« A se réduit polynomialement à B »), ce qui signifie que **B est au moins aussi difficile que A**.

> **Sens de la réduction :** pour prouver que le Δ-TSP est NP-Difficile, on réduit un problème NP-Complet *vers* le Δ-TSP — et non l'inverse.

---

### 3.2 Problème source : le Cycle Hamiltonien

Le problème du **Cycle Hamiltonien** est le suivant :

> Étant donné un graphe $G = (V, E)$ non pondéré, existe-t-il un cycle passant par chaque sommet exactement une fois ?

Ce problème est **NP-Complet** (Karp, 1972). On va montrer :

$$\text{Cycle Hamiltonien} \leq_p \Delta\text{-TSP}$$

---

### 3.3 Construction de la réduction

Soit $G = (V, E)$ une instance quelconque du Cycle Hamiltonien. On construit une instance du Δ-TSP comme suit :

**Graphe $G' = (V, E', w')$ :**
- Mêmes sommets que $G$
- $E'$ est **complet** : toutes les paires de sommets sont reliées
- Les poids sont définis par :

$$w'(\{v_i, v_j\}) = \begin{cases} 1 & \text{si } \{v_i, v_j\} \in E \\ 2 & \text{si } \{v_i, v_j\} \notin E \end{cases}$$

**Borne :** $k = |V| = n$

**Complexité de la transformation :** construire $G'$ requiert d'examiner toutes les paires de sommets, soit $O(n^2)$ — **polynomiale**.

---

### 3.4 Vérification de l'hypothèse métrique

Pour que cette réduction soit valide pour le **Δ-TSP**, il faut vérifier que $G'$ respecte l'inégalité triangulaire.

Pour tous sommets $v_i, v_j, v_k \in V$ :
- $w'(\{v_i, v_j\}) \leq 2$ (poids maximal)
- $w'(\{v_i, v_k\}) + w'(\{v_k, v_j\}) \geq 1 + 1 = 2$ (somme minimale)

Donc $w'(\{v_i, v_j\}) \leq w'(\{v_i, v_k\}) + w'(\{v_k, v_j\})$ est **toujours vérifié**.

L'instance construite est bien une instance valide du **Δ-TSP**.

---

### 3.5 Preuve de conservation de la réponse

**($\Rightarrow$) Si $G$ possède un cycle hamiltonien, alors le Δ-TSP répond « oui ».**

Soit $C = (v_{\sigma(0)}, v_{\sigma(1)}, \dots, v_{\sigma(n-1)}, v_{\sigma(0)})$ un cycle hamiltonien dans $G$. Toutes les arêtes de $C$ appartiennent à $E$, donc ont poids $1$ dans $G'$. Le coût total de $C$ dans $G'$ est :

$$\text{coût}(C) = n \times 1 = n = k$$

Le cycle $C$ est un cycle hamiltonien de $G'$ de coût $\leq k$ : le Δ-TSP répond **oui**.

**($\Leftarrow$) Si le Δ-TSP répond « oui », alors $G$ possède un cycle hamiltonien.**

Supposons qu'il existe un cycle hamiltonien $C'$ dans $G'$ de coût $\leq k = n$. Ce cycle visite $n$ sommets et emprunte $n$ arêtes. Chaque arête a poids $\geq 1$, donc le coût total est $\geq n$. Combiné avec coût $\leq n$, on obtient :

$$\text{coût}(C') = n \implies \text{toutes les arêtes ont poids } 1 \implies \text{toutes les arêtes appartiennent à } E$$

Le cycle $C'$ est donc un cycle hamiltonien valide dans $G$.

---

### 3.6 Conclusion

La réduction $\text{Cycle Hamiltonien} \leq_p \Delta\text{-TSP}$ est établie :
- La transformation est **polynomiale** en $O(n^2)$
- La réponse est **conservée dans les deux sens**

Le Δ-TSP est donc **NP-Difficile**. Combiné avec l'appartenance à NP (Partie 2) :

$$\boxed{\Delta\text{-TSP est NP-Complet}}$$

**Conséquence pratique :** sauf si $P = NP$, il n'existe pas d'algorithme polynomial garantissant la solution optimale. Pour des instances de taille réaliste, il faudra recourir à des **algorithmes heuristiques**.

In [4]:
import itertools

def reduction_hamiltonien_vers_tsp(sommets, aretes):
    """
    Réduction polynomiale : Cycle Hamiltonien -> Δ-TSP.

    Transforme une instance du Cycle Hamiltonien en instance du Δ-TSP :
    - Arête présente dans G  -> poids 1
    - Arête absente dans G   -> poids 2
    - Borne k = |V|

    Complexité : O(n²) — polynomiale.

    Paramètres :
        sommets : liste des sommets
        aretes  : ensemble de tuples (u, v) représentant les arêtes de G

    Retourne :
        distances : dict {(u, v): poids} du graphe complet G'
        k         : borne pour le problème de décision du Δ-TSP
    """
    n = len(sommets)

    # Normaliser les arêtes (u, v) en (min, max) pour la recherche
    aretes_norm = set()
    for u, v in aretes:
        aretes_norm.add((min(u, v), max(u, v)))

    # Construire le graphe complet G' avec poids 1 ou 2
    distances = {}
    for i in range(n):
        for j in range(i + 1, n):
            u, v = sommets[i], sommets[j]
            cle = (min(u, v), max(u, v))
            distances[cle] = 1 if cle in aretes_norm else 2

    k = n  # borne = nombre de sommets
    return distances, k


def meilleur_cycle(sommets, distances):
    """Recherche exhaustive du cycle hamiltonien de coût minimal."""
    meilleur_cout = float('inf')
    for perm in itertools.permutations(sommets[1:]):
        cycle = [sommets[0]] + list(perm)
        cout = sum(
            distances[(min(cycle[i], cycle[(i+1) % len(sommets)]),
                       max(cycle[i], cycle[(i+1) % len(sommets)]))]
            for i in range(len(sommets))
        )
        meilleur_cout = min(meilleur_cout, cout)
    return meilleur_cout


# ============================================================
# CAS 1 : Graphe AVEC cycle hamiltonien  (1-2-3-4-5-1)
# ============================================================
print("=" * 55)
print("CAS 1 : Graphe avec cycle hamiltonien")
print("=" * 55)

sommets_1 = [1, 2, 3, 4, 5]
aretes_1  = {(1, 2), (2, 3), (3, 4), (4, 5), (5, 1), (1, 3)}

dist_1, k_1 = reduction_hamiltonien_vers_tsp(sommets_1, aretes_1)

print(f"G  : sommets={sommets_1}")
print(f"     arêtes={aretes_1}")
print(f"G' : k = {k_1}")

# Vérifier le cycle hamiltonien connu 1-2-3-4-5-1
cycle_1 = [1, 2, 3, 4, 5]
cout_1 = sum(
    dist_1[(min(cycle_1[i], cycle_1[(i+1) % 5]),
            max(cycle_1[i], cycle_1[(i+1) % 5]))]
    for i in range(5)
)
print(f"\nCycle 1→2→3→4→5→1 : coût = {cout_1}")
print(f"Coût ≤ k ({k_1}) ? {'OUI' if cout_1 <= k_1 else 'NON'}")
print("=> G a un cycle hamiltonien  ET  Δ-TSP répond OUI ✓")

# ============================================================
# CAS 2 : Graphe SANS cycle hamiltonien
# ============================================================
print(f"\n{'=' * 55}")
print("CAS 2 : Graphe sans cycle hamiltonien")
print("=" * 55)

sommets_2 = [1, 2, 3, 4, 5]
aretes_2  = {(1, 2), (2, 3)}   # trop peu d'arêtes

dist_2, k_2 = reduction_hamiltonien_vers_tsp(sommets_2, aretes_2)

print(f"G  : sommets={sommets_2}")
print(f"     arêtes={aretes_2}")
print(f"G' : k = {k_2}")

cout_min_2 = meilleur_cycle(sommets_2, dist_2)
print(f"\nMeilleur cycle possible : coût = {cout_min_2}")
print(f"Coût ≤ k ({k_2}) ? {'OUI' if cout_min_2 <= k_2 else 'NON'}")
print("=> G n'a PAS de cycle hamiltonien  ET  Δ-TSP répond NON ✓")

# ============================================================
# CONCLUSION
# ============================================================
print(f"\n{'=' * 55}")
print("CONCLUSION")
print("=" * 55)
print("La réduction conserve la réponse dans les deux sens :")
print("  G a un cycle hamiltonien  <=>  Δ-TSP(G', k=n) répond OUI")
print("  Transformation en O(n²)   =>   polynomiale")
print("\n=> Cycle Hamiltonien ≤p Δ-TSP")
print("=> Le Δ-TSP est NP-Difficile")
print("=> Le Δ-TSP est NP-Complet (NP + NP-Difficile)")

CAS 1 : Graphe avec cycle hamiltonien
G  : sommets=[1, 2, 3, 4, 5]
     arêtes={(2, 3), (4, 5), (1, 2), (3, 4), (1, 3), (5, 1)}
G' : k = 5

Cycle 1→2→3→4→5→1 : coût = 5
Coût ≤ k (5) ? OUI
=> G a un cycle hamiltonien  ET  Δ-TSP répond OUI ✓

CAS 2 : Graphe sans cycle hamiltonien
G  : sommets=[1, 2, 3, 4, 5]
     arêtes={(2, 3), (1, 2)}
G' : k = 5

Meilleur cycle possible : coût = 8
Coût ≤ k (5) ? NON
=> G n'a PAS de cycle hamiltonien  ET  Δ-TSP répond NON ✓

CONCLUSION
La réduction conserve la réponse dans les deux sens :
  G a un cycle hamiltonien  <=>  Δ-TSP(G', k=n) répond OUI
  Transformation en O(n²)   =>   polynomiale

=> Cycle Hamiltonien ≤p Δ-TSP
=> Le Δ-TSP est NP-Difficile
=> Le Δ-TSP est NP-Complet (NP + NP-Difficile)


## Partie 4 : Conclusion et conséquences pratiques

### 4.1 Récapitulatif de la preuve

| Étape | Résultat | Méthode |
|-------|----------|---------|
| Modélisation | Le problème de tournée se ramène au **Δ-TSP** | Complétion par plus courts chemins (Dijkstra) |
| Appartenance à NP | Le Δ-TSP est **dans NP** | Algorithme certificat en $O(n)$ |
| NP-Difficulté | Le Δ-TSP est **NP-Difficile** | Réduction polynomiale depuis le Cycle Hamiltonien |
| **Conclusion** | Le Δ-TSP est **NP-Complet** | NP $\cap$ NP-Difficile |

---

### 4.2 Ce que NP-Complet signifie concrètement

La NP-Complétude du Δ-TSP implique que, **sauf si $P = NP$** (ce qui est considéré comme très improbable) :

- Il n'existe **aucun algorithme polynomial** qui garantit de trouver la tournée optimale
- Tout algorithme **exact** (garantissant l'optimum) a une complexité au moins exponentielle
- L'approche force brute en $O(n!)$ et l'algorithme de Held-Karp en $O(n^2 \cdot 2^n)$ sont les meilleures solutions exactes connues

À titre indicatif, pour $n = 20$ villes à $10^9$ opérations/seconde :

| Algorithme | Complexité | Temps estimé |
|------------|-----------|--------------|
| Force brute | $O(n!)$ | $\approx 3{,}9$ années |
| Held-Karp | $O(n^2 \cdot 2^n)$ | $\approx 4$ secondes |
| Heuristique | $O(n^2)$ | $< 1$ ms |

> **Remarque :** NP-Complet ne signifie pas que le problème est insoluble. On peut toujours trouver une solution — la question est le temps que cela prend pour de grandes instances.

---

### 4.3 Complexité spatiale

La complexité spatiale du Δ-TSP est **polynomiale** : stocker le graphe complet $G'$ requiert $O(n^2)$ cases mémoire (la matrice des distances). Avec les capacités des calculateurs modernes, ce n'est pas un obstacle pratique — c'est la complexité **temporelle** qui constitue le véritable enjeu.

---

### 4.4 Choix de la stratégie algorithmique

Face à un problème NP-Complet, deux stratégies s'offrent à nous :

| | Approche exacte | Approche heuristique |
|-|----------------|---------------------|
| **Optimalité** | Garantie | Non garantie |
| **Complexité** | Exponentielle | Polynomiale |
| **Taille max** | $n \lesssim 20$ | $n$ en milliers |
| **Exemples** | Force brute, Held-Karp | Plus proche voisin, 2-opt, recuit simulé |

Dans le cadre du projet ADEME, les tournées de livraison impliquent potentiellement des dizaines ou centaines de villes. Une approche exacte serait donc impraticable. On retient une **approche heuristique**, qui offre une solution de bonne qualité en temps raisonnable, au prix d'une garantie d'optimalité abandonnée.
```